# Decision Transformer — Reinforcement Learning via Sequence Modeling (2021)

_Papers · Reinforcement Learning_

**Recast reinforcement learning as next-token prediction: feed a causal transformer (desired-return, state, action) tokens and let it autoregressively output the action that hits the return you asked for.**

---

This notebook is a practice scaffold for the **Decision Transformer — Reinforcement Learning via Sequence Modeling (2021)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch (Colab)

In [ ]:
# torch is preinstalled in Colab. No gym needed: the toy environment is a few lines.
import torch
import torch.nn as nn
import numpy as np
import random

torch.manual_seed(0); np.random.seed(0); random.seed(0)

# --- 0. Sanity-check the worked return-to-go example. ---
def returns_to_go(rewards):                 # R_hat_t = sum_{t'=t}^{T} r_t'   (Section 3)
    out = [0.0] * len(rewards); acc = 0.0
    for t in reversed(range(len(rewards))): # one backward pass = suffix sums
        acc += rewards[t]; out[t] = acc
    return out

rew_demo = [0, 0, 0, 1, 1, 1, 1, 0, 0, 0]
print("rewards     :", rew_demo)
print("return-to-go:", [int(x) for x in returns_to_go(rew_demo)])   # [4,4,4,4,3,2,1,0,0,0]


# --- 1. Toy OFFLINE environment: 1-D grid of length N. ---
# State = position 0..N-1. Action 0=LEFT(-1), 1=RIGHT(+1). Reward +1 at the
# rightmost cell, else 0. Optimal: rush right and stay -> high return.
N, T = 5, 10
def env_step(s, a):
    s2 = max(0, min(N - 1, s + (1 if a == 1 else -1)))
    return s2, (1.0 if s2 == N - 1 else 0.0)

def gen_traj(p_right):                       # p_right = probability of going RIGHT
    s, states, actions, rewards = 0, [], [], []
    for _ in range(T):
        a = 1 if random.random() < p_right else 0
        s2, r = env_step(s, a)
        states.append(s); actions.append(a); rewards.append(r); s = s2
    return states, actions, rewards

# A fixed, MIXED-quality log: expert-ish, medium, and poor trajectories.
dataset = ([gen_traj(0.95) for _ in range(120)] +
           [gen_traj(0.50) for _ in range(120)] +
           [gen_traj(0.15) for _ in range(120)])
rets = [sum(tr[2]) for tr in dataset]
print("dataset returns: min %.0f  mean %.2f  max %.0f" % (min(rets), np.mean(rets), max(rets)))

R = torch.tensor([returns_to_go(tr[2]) for tr in dataset], dtype=torch.float32)
S = torch.tensor([tr[0] for tr in dataset], dtype=torch.long)
A = torch.tensor([tr[1] for tr in dataset], dtype=torch.long)


# --- 2. Tiny Decision Transformer (Track B: nn primitives). ---
D = 32
class TinyDT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed_rtg    = nn.Linear(1, D)        # return-to-go token (linear, Section 3)
        self.embed_state  = nn.Embedding(N, D)     # state token
        self.embed_action = nn.Embedding(2, D)     # action token
        self.embed_t      = nn.Embedding(T, D)     # per-TIMESTEP position (Section 3)
        self.ln = nn.LayerNorm(D)
        layer = nn.TransformerEncoderLayer(d_model=D, nhead=4, dim_feedforward=64,
                                           batch_first=True, activation='gelu', dropout=0.0)
        self.blocks = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Linear(D, 2)                # predict action logits

    def forward(self, rtgs, states, actions):
        B, Tt = states.shape
        te = self.embed_t(torch.arange(Tt, device=states.device))[None]      # (1,T,D)
        r = self.ln(self.embed_rtg(rtgs.unsqueeze(-1)) + te)                  # add position per timestep
        s = self.ln(self.embed_state(states)          + te)
        a = self.ln(self.embed_action(actions)        + te)
        tok = torch.stack([r, s, a], dim=2).reshape(B, Tt * 3, D)            # R0,s0,a0,R1,s1,a1,...
        L = Tt * 3
        mask = torch.triu(torch.ones(L, L, device=states.device), 1).bool()  # causal mask
        h = self.blocks(tok, mask=mask)
        return self.head(h[:, 1::3, :])             # action read off STATE token positions -> (B,T,2)

def train(rtg_tensor, steps=300):
    net = TinyDT(); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()                   # discrete actions -> cross-entropy (Section 3)
    for _ in range(steps):
        idx = torch.randperm(len(dataset))[:64]
        logits = net(rtg_tensor[idx], S[idx], A[idx])
        loss = lossf(logits.reshape(-1, 2), A[idx].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return net

net = train(R)


# --- 3. Return-conditioned rollout (Algorithm 1): seed target, then rtg -= reward. ---
@torch.no_grad()
def rollout(model, target_return, force_zero_rtg=False, n_eps=200):
    total = 0.0
    for ep in range(n_eps):
        s, rtg_t = 0, float(target_return)
        states, actions, rtgs, ep_ret = [], [], [], 0.0
        for t in range(T):
            states.append(s); rtgs.append(0.0 if force_zero_rtg else rtg_t)
            logits = model(torch.tensor([rtgs]), torch.tensor([states]),
                           torch.tensor([actions + [0]]))      # placeholder last action
            a = int(logits[0, t].argmax())
            actions.append(a)
            s, r = env_step(s, a)
            ep_ret += r; rtg_t -= r                            # decrement the target
        total += ep_ret
    return total / n_eps

print("\nReturn conditioning (achieved return tracks the target you ask for):")
for tgt in [2, 5, 8, 10]:
    print("  target=%2d  ->  achieved %.2f" % (tgt, rollout(net, tgt)))
# target= 2 -> 2.00 ; target= 5 -> 5.00 ; target>=8 -> 7.00 (env max reachable from start)


# --- 4. ABLATION: retrain with the return token forced to 0 -> steering disappears. ---
net_ablate = train(torch.zeros_like(R))
print("\nABLATION (return-to-go forced to 0 -> plain cloning, no steering):")
for tgt in [2, 5, 8, 10]:
    print("  target=%2d  ->  achieved %.2f" % (tgt, rollout(net_ablate, tgt, force_zero_rtg=True)))
# All targets -> ~0.00: with no return signal the model cannot be steered.
# (Exact numbers vary by hardware/seed; our small run, not the paper's.)

## Visualize the data & results

_Does conditioning a Decision Transformer on a higher TARGET return make it actually earn a higher return? We train one tiny DT on a fixed mixed-quality offline grid dataset, roll it out conditioning on a sweep of target returns, and compare against an ablated copy whose return-to-go token is forced to 0 (steering removed)._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Toy 1-D grid: reward +1 at the rightmost cell; mixed offline dataset of
# expert / medium / poor trajectories.
#
# Train DT: tokens (return-to-go, state, action) -> causal Transformer -> predict action
#   loss = cross_entropy(predicted_action, data_action)         # Section 3, no value loss
#
# Rollout, conditioned on a target return (Algorithm 1):
#   rtg = target
#   for each step:  a = argmax DT(rtg, state); observe r; rtg -= r   # the decrement
#   achieved = sum of rewards
#
# Sweep target in {1,...,10}:
#   return-conditioned  -> achieved tracks target, saturating at env max 7   (green)
#   ablation (rtg := 0) -> achieved flat ~0, target ignored                  (orange)
# (Numbers are from our own run; the paper reports Atari / D4RL Gym / Key-to-Door
#  results, not these toy-grid values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked return-to-go. An episode earns rewards
            $r = [0,0,1,0,1,1]$ over $T = 6$ steps. Compute the return-to-go vector $\hat R$ and state the
            target return you would condition on at step 1.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Start an accumulator at $0$ and walk from the last step backward, adding each reward. — _$\hat R_t = \sum_{t'=t}^{T} r_{t'}$ is a suffix sum; one backward pass computes all of them._
- $\hat R_6 = 1$, $\hat R_5 = 1+1 = 2$, $\hat R_4 = 0+2 = 2$, $\hat R_3 = 1+2 = 3$, $\hat R_2 = 0+3 = 3$, $\hat R_1 = 0+3 = 3$. — _Each step adds that step's reward to the running suffix total._
- Read off $\hat R_1$ as the total episode return. — _The first return-to-go equals the sum of all rewards — that is the number you would feed as the target to reproduce this episode._

**Answer:** $\hat R = [3, 3, 3, 2, 2, 1]$, and the target return at step 1 is $\hat R_1 = 3$ (the total
                 reward of the episode). Notice $\hat R$ drops by exactly the reward earned at each step &mdash;
                 the same rule as the test-time decrement $\hat R_{t+1} = \hat R_t - r_t$.

</details>

**Problem 2.** The return-conditioning result. After training on the mixed toy-grid dataset, you roll the
            model out conditioning on targets $2$, $5$, and $8$ (the environment's reachable maximum is $7$).
            What do you expect the achieved returns to be, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For each target, seed $\hat R_1 = $ target and roll out, decrementing $\hat R$ by each earned reward. — _Conditioning is the only thing that changes between runs — an honest test of the steering mechanism._
- Compare achieved return to the target. — _Figure 4 (Section 5.2) claims the two are highly correlated._
- Note the saturation when the target exceeds what the environment can pay. — _The model can only reproduce returns the data could actually achieve; it cannot exceed the environment's ceiling._

**Answer:** Achieved return tracks the target: in our run target $2 \to 2.00$, target $5 \to 5.00$,
                 and target $8 \to 7.00$ (it saturates at the reachable maximum of $7$). This reproduces the
                 paper's qualitative Figure-4 result &mdash; the number you condition on steers the behavior &mdash;
                 with the natural ceiling that you cannot ask for more return than the environment can pay.
                 (Our small run, not the paper's numbers.)

</details>

**Problem 3.** The ablation. You retrain an identical Decision Transformer but force the return-to-go token
            to $0$ at every step (removing the return conditioning). You re-run the target sweep. Predict the
            achieved return as a function of the target, and explain what the ablation isolates.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: replace every $\hat R_t$ with $0$, keeping the architecture, data, states, and actions identical. — _An honest ablation varies a single component — here, the return token — so any change in behavior is attributable to it._
- Re-train and roll out for targets $2, 5, 8$ (the conditioning input is now ignored / constant). — _If the model never saw an informative return, the target you feed cannot carry meaning._
- Observe that the achieved return no longer depends on the target. — _Without the return signal the model is plain behavioral cloning of a mixed dataset — it cannot be steered._

**Answer:** The achieved return becomes flat and target-insensitive: in our run it collapses to
                 $0.00$ for every target ($2, 5, 8$ all give $\approx 0$), because cloning a mixed-quality
                 dataset with no return signal averages into aimless behavior that rarely reaches the goal. The
                 ablation isolates the paper's core claim: the return-to-go token is what makes Decision
                 Transformer controllable. Remove it and you are left with undirected imitation.
                 (Our small run, not the paper's numbers.)

</details>